# Lab5.2 — Primul resize în Programmable Logic


Executăm aceeași operație de resize ca în Lab4, dar acum pixelii sunt procesați de acceleratorul hardware din PL.


Pregătim și localizăm Overlay-ul Resize.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

OVERLAY_DIR = Path("overlay")
OVERLAY_DIR.mkdir(exist_ok=True)
LOCAL_BIT = OVERLAY_DIR / "resizer.bit"
LOCAL_HWH = OVERLAY_DIR / "resizer.hwh"

def find_resizer_overlay():
    if LOCAL_BIT.exists() and LOCAL_HWH.exists():
        return LOCAL_BIT

    try:
        import pynq_helloworld
        pkg = Path(pynq_helloworld.__file__).resolve().parent / "notebooks" / "edge"
        pkg_bit = pkg / "resizer.bit"
        pkg_hwh = pkg / "resizer.hwh"
        if pkg_bit.exists() and pkg_hwh.exists():
            return pkg_bit
    except ImportError:
        pass

    urls = {
        LOCAL_BIT: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.bit",
        LOCAL_HWH: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.hwh",
    }
    for path, url in urls.items():
        if not path.exists():
            print(f"Downloading {path.name} ...")
            urlretrieve(url, path)

    return LOCAL_BIT

RESIZER_BIT = find_resizer_overlay()
print("Overlay:", RESIZER_BIT)

Stabilim calea către aceeași imagine folosită în Lab4.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("../Lab4/images/fruits.jpg")
if not IMAGE_PATH.exists():
    raise FileNotFoundError("Nu a fost găsită imaginea ../Lab4/images/fruits.jpg")
print("Image:", IMAGE_PATH)

Încărcăm bibliotecile, Overlay-ul și imaginea de test.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay(str(RESIZER_BIT))
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("Imaginea nu a putut fi încărcată.")

old_h, old_w, channels = image_bgr.shape
new_w, new_h = 320, 240
print(f"Input : {old_w} × {old_h}")
print(f"Output: {new_w} × {new_h}")

Alocăm două buffere accesibile DMA-ului: unul pentru imaginea de intrare și unul pentru rezultat.


In [ ]:
in_buffer = allocate(shape=(old_h, old_w, 3), dtype=np.uint8)
out_buffer = allocate(shape=(new_h, new_w, 3), dtype=np.uint8)

np.copyto(in_buffer, image_bgr)
out_buffer[:] = 0

print("Input buffer :", in_buffer.shape, in_buffer.nbytes, "bytes")
print("Output buffer:", out_buffer.shape, out_buffer.nbytes, "bytes")

Scriem în registrele acceleratorului dimensiunile imaginii sursă și ale imaginii rezultate.


In [ ]:
resizer.register_map.src_rows = old_h
resizer.register_map.src_cols = old_w
resizer.register_map.dst_rows = new_h
resizer.register_map.dst_cols = new_w

print(resizer.register_map)

Definim funcția care pornește transferul DMA, acceleratorul și așteaptă finalizarea procesării.


In [ ]:
def run_kernel():
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)
    resizer.write(0x00, 0x81)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

Executăm acceleratorul o dată și copiem rezultatul într-un array NumPy obișnuit.


In [ ]:
run_kernel()
result_bgr = np.array(out_buffer)
print("Hardware resize completed:", result_bgr.shape)

Afișăm originalul și rezultatul hardware **la scară relativă**, astfel încât reducerea rezoluției să fie vizibilă.


In [ ]:
original_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

h1, w1, _ = original_rgb.shape
h2, w2, _ = result_rgb.shape
gap = 30
canvas_h = max(h1, h2)
canvas_w = w1 + gap + w2
canvas = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255
canvas[:h1, :w1] = original_rgb
canvas[:h2, w1+gap:w1+gap+w2] = result_rgb

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(canvas)
ax.set_ylim(canvas_h, -45)
ax.text(w1/2, -18, f"Original: {w1} × {h1}", ha="center", va="center")
ax.text(w1+gap+w2/2, -18, f"PL Resize: {w2} × {h2}", ha="center", va="center")
ax.axis("off")
plt.tight_layout()
plt.show()

Eliberăm bufferele alocate după terminarea experimentului.


In [ ]:
del in_buffer, out_buffer


## Întrebări

1. Ce date sunt transferate de canalul MM2S?
2. Ce date sunt transferate de canalul S2MM?
3. De ce bufferul de ieșire are mai puțini bytes decât bufferul de intrare?
